# 第 22 课｜给果蝇一个世界：把 loop 闭合起来

当 neural system 的 output 会改变 environment，而 environment 又产生下一轮 input 时，实验就从固定回放变成了闭环。

今天只问一个问题：

> **closed-loop experiment 与固定 replay 的根本区别是什么？**

本课主要新概念：**closed loop——output 会改变未来 input。**

本课 toy world 只是工程教学模型，**不声称代表真实果蝇行为，也不声称代表 MaleCNS 的真实 sensory semantics。**

## 1. 概念账本

**已经知道：** host input、spike output、deterministic replay、connectome structure。

**今天学习：** **closed loop（闭环）**：系统 output 改变 environment state，environment 又改变后续 sensory input。

**支持角色：** **sensory encoder** 把 environment observation 映射成 neural stimulation；**output decoder** 把选定 neural activity 映射成 action。

**只预告：** 有生物依据的 sensory mapping 与 descending-neuron behavior mapping。

## 2. open-loop replay 与 closed loop

open-loop replay 中，input sequence 在运行前就固定好了。

closed loop 中，(t+1) 时刻的 input 会依赖系统在 (t) 时刻做了什么。

这个 feedback 才是今天的主要新概念。encoder 与 decoder 被显式画成边界，是为了让人工假设可见。

## 3. closed-loop 图

<div style="max-width:860px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 860 360" role="img" aria-label="closed loop environment neural system action" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="l22-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2f5f3f"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="19" text-anchor="middle">
    <rect x="25" y="110" width="150" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="100" y="140" fill="#1f2d24">environment</text><text x="100" y="165" fill="#1f2d24">state</text>
    <rect x="195" y="110" width="150" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="270" y="140" fill="#1f2d24">sensory</text><text x="270" y="165" fill="#1f2d24">encoder</text>
    <rect x="365" y="110" width="150" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="440" y="140" fill="#1f2d24">neural</text><text x="440" y="165" fill="#1f2d24">system</text>
    <rect x="535" y="110" width="150" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="610" y="140" fill="#1f2d24">output</text><text x="610" y="165" fill="#1f2d24">decoder</text>
    <rect x="705" y="110" width="130" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="770" y="153" fill="#1f2d24">action</text>
  </g>
  <g fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l22-arrow)">
    <path d="M175 145 L195 145"/><path d="M345 145 L365 145"/><path d="M515 145 L535 145"/><path d="M685 145 L705 145"/>
    <path d="M770 180 C770 300,100 300,100 180"/>
  </g>
</svg>
</div>

## 4. Run：一个 deterministic 一维 toy world

这个 teaching world 的规则是：

- target 固定在 position 3；
- target 在右侧时，encoder 产生 `right`；
- neural-system stub 直接把这个 event 转成 output event；
- decoder 把 `right` 变成 action +1；
- environment 更新 position。

这些规则都故意写得很简单，而且都是人工定义的。

In [ ]:
position = 0
target = 3
trace = [position]

for step in range(5):
    if position < target:
        sensory_event = "right"
    elif position > target:
        sensory_event = "left"
    else:
        sensory_event = "at_target"

    output_event = sensory_event  # teaching stub, not a biological claim

    action = {
        "right": 1,
        "left": -1,
        "at_target": 0,
    }[output_event]

    position += action
    trace.append(position)
    print(step, sensory_event, action, position)

print("position trace:", trace)

## 5. Observe

position 到达 target 后，下一次 observation 变成 `at_target`，action 变成 0，后续 input 因此和固定 replay 不同。

这种“之前的 output 改变了未来 input”的依赖，就是 closed loop。

## 6. 工程假设必须可见

项目工程文档要求人工 sensory/output mapping 必须显式记录。

在这个 toy world 里，下列内容全都是假设：

- environment variable 代表什么；
- “target 在右边”由哪个 sensory event 表示；
- 哪个 neural output 映射成 +1 movement；
- action 怎样改变 world。

未来真正做 biological mapping 时，需要自己的 evidence，不能悄悄继承这些 toy rules。

## 7. closed loop 仍然可以 deterministic replay

closed loop 不等于“不可重复”。

如果 model version、network image、initial world state、mapping rule、input randomness 与 PRNG seed 都固定，那么完整 trajectory 应该可 replay。这正是 full-system determinism requirement 的意义。

## 8. Try It

把 target 从 3 改成 -2。运行前先预测 position trace。这个 case 会强制系统向左移动，因此能直接暴露“只实现了向右 +1 分支”的错误。

再改变 initial position。哪些 future input 会因为前一次 action 改变 environment 而改变？

## 9. 作业

[第 22 课作业：生成 deterministic closed-loop position trace](../../exercises/zh/22_closed_loop_world.ipynb)

## 10. AI Task

让 AI 提议一个更丰富的二维 toy environment。要求它把每一个人工选择的 sensory/action mapping 与 biological fact 分开列出。

## 11. Human Check

解释 fixed input replay 与 closed-loop experiment 的区别。哪些假设属于 sensory encoder，哪些属于 output decoder，哪些属于 environment？

## 12. Engineering Handoff

对应 `RMD-023 / RMD-024 / RMD-025`、`MOD-012 sensory_encoder`、`MOD-013 output_decoder` 与 `MOD-014 telemetry`。人工 mapping 在进入正式 experiment 前必须显式文档化。

## 13. Project Trace

- Lesson：`LSN-022`
- 映射：`RMD-023~025`
- 需求路径：`TRACE-IO-001`
- 可重复性路径：`TRACE-F-001`
- determinism oracle：`T-016`

## 14. Exit Ticket

你能够解释 loop 为什么是 closed，识别 encoder/decoder/environment 的人工假设，并说明为了 replay 同一 trajectory 必须固定哪些东西。